downloading ucimlrepo<br>
since we will be using a health disease dataset from https://archive.ics.uci.edu/dataset/45/heart+disease

In [ ]:
!pip install ucimlrepo

importing the modules.

In [ ]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

## importing the dataset from the site.

In [ ]:
# fetch dataset
heart_disease = fetch_ucirepo(id=45)

# data (as pandas dataframes)
X = heart_disease.data.features
y = heart_disease.data.targets
df = pd.concat([X, y], axis=1)

# metadata
print(heart_disease.metadata)
# variable information
print(heart_disease.variables)

{'uci_id': 45, 'name': 'Heart Disease', 'repository_url': 'https://archive.ics.uci.edu/dataset/45/heart+disease', 'data_url': 'https://archive.ics.uci.edu/static/public/45/data.csv', 'abstract': '4 databases: Cleveland, Hungary, Switzerland, and the VA Long Beach', 'area': 'Health and Medicine', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 303, 'num_features': 13, 'feature_types': ['Categorical', 'Integer', 'Real'], 'demographics': ['Age', 'Sex'], 'target_col': ['num'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1989, 'last_updated': 'Fri Nov 03 2023', 'dataset_doi': '10.24432/C52P4X', 'creators': ['Andras Janosi', 'William Steinbrunn', 'Matthias Pfisterer', 'Robert Detrano'], 'intro_paper': {'ID': 231, 'type': 'NATIVE', 'title': 'International application of a new probability algorithm for the diagnosis of coronary artery disease.', 'authors': 'R. Detrano, A. Jánosi, W. Steinbrunn, M

In [ ]:
print(df.info())
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
 13  num       303 non-null    int64  
dtypes: float64(3), int64(11)
memory usage: 33.3 KB
None
              age         sex          cp    trestbps        chol         fbs  \
count  303.000000  303.000000  303.000000  303.000000  303.000000  303.000000   
mean    54.438944    0.679868    3.158416  131.68976

## we will now clean and process the data.

In [ ]:
#checking if any column has mission values
print(df.isnull().sum())

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
num         0
dtype: int64


#### column `ca` and `thal` has missing values <br>
we will fill `ca`(numerical) with median values, and `thal`(categorical) with mode values.

In [ ]:
df['ca'] = df['ca'].fillna(df['ca'].median()) #median for ca
print(f"null values in ca is replaced with {df['ca'].median()}")
df['thal'] = df['thal'].fillna(df['thal'].mode()[0]) #mode for thal
print(f"null values in thal is replaced with {df['thal'].mode()[0]}")

null values in ca is replaced with 0.0
null values in thal is replaced with 3.0


In [ ]:
print("null values in:")
print(f"ca: {df['ca'].isnull().sum()}")
print(f"thal: {df['thal'].isnull().sum()}")

null values in:
ca: 0
thal: 0


## Aggregation.

In [ ]:
aggregation = df.groupby('num').agg({
    'age': 'mean',
    'trestbps': 'mean',
    'chol': 'mean',
    'thalach': 'mean'
})

print(aggregation)

           age    trestbps        chol     thalach
num                                               
0    52.585366  129.250000  242.640244  158.378049
1    55.381818  133.254545  249.109091  145.927273
2    58.027778  134.194444  259.277778  135.583333
3    56.000000  135.457143  246.457143  132.057143
4    59.692308  138.769231  253.384615  140.615385


In [ ]:
aggregation = df.groupby('num').agg({
    'age': ['mean', 'min', 'max'],
    'chol': ['mean', 'median'],
    'trestbps': ['mean', 'max']
})

print(aggregation)

           age                chol           trestbps     
          mean min max        mean median        mean  max
num                                                       
0    52.585366  29  76  242.640244  234.5  129.250000  180
1    55.381818  35  70  249.109091  249.0  133.254545  192
2    58.027778  42  69  259.277778  254.0  134.194444  180
3    56.000000  39  70  246.457143  256.0  135.457143  200
4    59.692308  38  77  253.384615  231.0  138.769231  165


#Discretization

In [ ]:
chol_bins = [0, 200, 240, 1000]

chol_labels = [
    'Normal',
    'Borderline',
    'High'
]

df['chol_category'] = pd.cut(
    df['chol'],
    bins=chol_bins,
    labels=chol_labels
)

print(df[['chol', 'chol_category']].head())

   chol chol_category
0   233    Borderline
1   286          High
2   229    Borderline
3   250          High
4   204    Borderline


#Binarization

In [ ]:
df['high_chol'] = (df['chol'] >= 240).astype(int)
df['high_bp'] = (df['trestbps'] >= 140).astype(int)
df['heart_disease'] = (df['num'] > 0).astype(int)
print(df[['chol', 'trestbps', 'high_chol', 'high_bp', 'heart_disease']].head())

   chol  trestbps  high_chol  high_bp  heart_disease
0   233       145          0        1              0
1   286       160          1        1              1
2   229       120          0        0              1
3   250       130          1        0              0
4   204       130          0        0              0


#Sampling
selecting a subset of the entire dataset.

In [ ]:
sample = df.sample(
    n=100,
    random_state=42
)

print(sample)

     age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  \
179   53    1   3       130   246    1        2      173      0      0.0   
228   54    1   4       110   206    0        2      108      1      0.0   
111   56    1   4       125   249    1        2      144      1      1.2   
246   58    1   4       100   234    0        0      156      0      0.1   
60    51    0   4       130   305    0        0      142      1      1.2   
..   ...  ...  ..       ...   ...  ...      ...      ...    ...      ...   
163   58    0   4       100   248    0        2      122      0      1.0   
155   70    1   4       130   322    0        2      109      0      2.4   
97    60    0   4       150   258    0        2      157      0      2.6   
68    59    1   4       170   326    0        2      140      1      3.4   
229   66    1   4       112   212    0        2      132      1      0.1   

     slope   ca  thal  num chol_category  high_chol  high_bp  heart_disease  
179      